In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the cleaned hourly data
df = pd.read_csv('../data/processed/hourly_data.csv', index_col='time', parse_dates=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Nulls: {df.isnull().sum().sum()}")

Shape: (8399, 27)
Date range: 2016-01-01 00:00:00 to 2016-12-15 22:00:00
Nulls: 0


In [2]:
# Time decomposition features
df['hour'] = df.index.hour
df['day_of_week'] = df.index.dayofweek        # 0=Monday, 6=Sunday
df['month'] = df.index.month
df['day_of_year'] = df.index.dayofyear
df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)
df['is_nighttime'] = ((df.index.hour >= 22) | (df.index.hour <= 5)).astype(int)

# Cyclical encoding — preserves circular nature of time
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

print("Time features added:")
time_features = ['hour', 'day_of_week', 'month', 'day_of_year',
                 'is_weekend', 'is_nighttime',
                 'hour_sin', 'hour_cos', 'day_sin', 'day_cos']

print(df[time_features].head(8).to_string())

Time features added:
                     hour  day_of_week  month  day_of_year  is_weekend  is_nighttime  hour_sin      hour_cos   day_sin   day_cos
time                                                                                                                            
2016-01-01 00:00:00     0            4      1            1           0             1  0.000000  1.000000e+00 -0.433884 -0.900969
2016-01-01 01:00:00     1            4      1            1           0             1  0.258819  9.659258e-01 -0.433884 -0.900969
2016-01-01 02:00:00     2            4      1            1           0             1  0.500000  8.660254e-01 -0.433884 -0.900969
2016-01-01 03:00:00     3            4      1            1           0             1  0.707107  7.071068e-01 -0.433884 -0.900969
2016-01-01 04:00:00     4            4      1            1           0             1  0.866025  5.000000e-01 -0.433884 -0.900969
2016-01-01 05:00:00     5            4      1            1           0      

In [3]:
# Lag features — looking backward in time at the target variable
target = 'use [kW]'

df['lag_1h']   = df[target].shift(1)
df['lag_2h']   = df[target].shift(2)
df['lag_3h']   = df[target].shift(3)
df['lag_24h']  = df[target].shift(24)
df['lag_48h']  = df[target].shift(48)
df['lag_168h'] = df[target].shift(168)   # same hour last week

print("Lag features added:")
lag_features = ['lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'lag_48h', 'lag_168h']
print(df[lag_features].head(10).to_string())
print(f"\nNulls introduced by lags:")
print(df[lag_features].isnull().sum())

Lag features added:
                       lag_1h    lag_2h    lag_3h  lag_24h  lag_48h  lag_168h
time                                                                         
2016-01-01 00:00:00       NaN       NaN       NaN      NaN      NaN       NaN
2016-01-01 01:00:00  1.044130       NaN       NaN      NaN      NaN       NaN
2016-01-01 02:00:00  0.918167  1.044130       NaN      NaN      NaN       NaN
2016-01-01 03:00:00  0.714736  0.918167  1.044130      NaN      NaN       NaN
2016-01-01 04:00:00  0.960013  0.714736  0.918167      NaN      NaN       NaN
2016-01-01 05:00:00  0.639836  0.960013  0.714736      NaN      NaN       NaN
2016-01-01 06:00:00  1.219416  0.639836  0.960013      NaN      NaN       NaN
2016-01-01 07:00:00  0.798747  1.219416  0.639836      NaN      NaN       NaN
2016-01-01 08:00:00  0.537055  0.798747  1.219416      NaN      NaN       NaN
2016-01-01 09:00:00  0.358377  0.537055  0.798747      NaN      NaN       NaN

Nulls introduced by lags:
lag_1h        1
l

In [4]:
# Rolling statistics on the target variable
# min_periods=1 means we calculate even with partial windows at the start
df['rolling_mean_3h']  = df[target].shift(1).rolling(window=3,  min_periods=1).mean()
df['rolling_mean_24h'] = df[target].shift(1).rolling(window=24, min_periods=1).mean()
df['rolling_std_24h']  = df[target].shift(1).rolling(window=24, min_periods=1).std()
df['rolling_max_24h']  = df[target].shift(1).rolling(window=24, min_periods=1).max()

print("Rolling features added:")
rolling_features = ['rolling_mean_3h', 'rolling_mean_24h', 
                    'rolling_std_24h', 'rolling_max_24h']
print(df[rolling_features].head(10).to_string())
print(f"\nNulls in rolling features:")
print(df[rolling_features].isnull().sum())

Rolling features added:
                     rolling_mean_3h  rolling_mean_24h  rolling_std_24h  rolling_max_24h
time                                                                                    
2016-01-01 00:00:00              NaN               NaN              NaN              NaN
2016-01-01 01:00:00         1.044130          1.044130              NaN         1.044130
2016-01-01 02:00:00         0.981148          0.981148         0.089069         1.044130
2016-01-01 03:00:00         0.892344          0.892344         0.166208         1.044130
2016-01-01 04:00:00         0.864305          0.909261         0.139863         1.044130
2016-01-01 05:00:00         0.771528          0.855376         0.170848         1.044130
2016-01-01 06:00:00         0.939755          0.916050         0.213164         1.219416
2016-01-01 07:00:00         0.886000          0.899292         0.199578         1.219416
2016-01-01 08:00:00         0.851739          0.854012         0.224818         1.2194

In [5]:
# Record shape before dropping
rows_before = len(df)

# Drop rows with any nulls — these are the warmup rows at the start
# that don't have enough history for lag and rolling features
df_features = df.dropna()

rows_after = len(df_features)
rows_dropped = rows_before - rows_after

print(f"Rows before dropping nulls: {rows_before}")
print(f"Rows dropped (warmup period): {rows_dropped}")
print(f"Rows remaining: {rows_after}")
print(f"Date range after drop: {df_features.index.min()} to {df_features.index.max()}")
print(f"Nulls remaining: {df_features.isnull().sum().sum()}")
print(f"\nFinal feature matrix shape: {df_features.shape}")
print(f"\nAll columns ({len(df_features.columns)}):")
for col in df_features.columns:
    print(f"  {col}")

Rows before dropping nulls: 8399
Rows dropped (warmup period): 168
Rows remaining: 8231
Date range after drop: 2016-01-08 00:00:00 to 2016-12-15 22:00:00
Nulls remaining: 0

Final feature matrix shape: (8231, 47)

All columns (47):
  use [kW]
  House overall [kW]
  Dishwasher [kW]
  Furnace 1 [kW]
  Furnace 2 [kW]
  Home office [kW]
  Fridge [kW]
  Wine cellar [kW]
  Garage door [kW]
  Kitchen 12 [kW]
  Kitchen 14 [kW]
  Barn [kW]
  Well [kW]
  Microwave [kW]
  Living room [kW]
  Solar [kW]
  temperature
  humidity
  visibility
  apparentTemperature
  pressure
  windSpeed
  cloudCover
  windBearing
  precipIntensity
  dewPoint
  precipProbability
  hour
  day_of_week
  month
  day_of_year
  is_weekend
  is_nighttime
  hour_sin
  hour_cos
  day_sin
  day_cos
  lag_1h
  lag_2h
  lag_3h
  lag_24h
  lag_48h
  lag_168h
  rolling_mean_3h
  rolling_mean_24h
  rolling_std_24h
  rolling_max_24h


In [6]:
# Define exactly which columns are model features vs target vs dropped
TARGET = 'use [kW]'

# These are raw appliance readings — not model features
# The model predicts total consumption, it doesn't get to peek at sub-readings
# during inference. We keep them in the saved file for reference only.
APPLIANCE_COLS = [
    'House overall [kW]', 'Dishwasher [kW]', 'Furnace 1 [kW]',
    'Furnace 2 [kW]', 'Home office [kW]', 'Fridge [kW]',
    'Wine cellar [kW]', 'Garage door [kW]', 'Kitchen 12 [kW]',
    'Kitchen 14 [kW]', 'Barn [kW]', 'Well [kW]', 'Microwave [kW]',
    'Living room [kW]', 'Solar [kW]'
]

# These are the actual model input features
FEATURE_COLS = [
    # Time features
    'hour', 'day_of_week', 'month', 'day_of_year',
    'is_weekend', 'is_nighttime',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    # Lag features
    'lag_1h', 'lag_2h', 'lag_3h',
    'lag_24h', 'lag_48h', 'lag_168h',
    # Rolling features
    'rolling_mean_3h', 'rolling_mean_24h',
    'rolling_std_24h', 'rolling_max_24h',
    # Weather features
    'temperature', 'humidity', 'windSpeed',
    'cloudCover', 'precipIntensity', 'dewPoint',
    'pressure', 'visibility', 'apparentTemperature',
    'windBearing', 'precipProbability'
]

print(f"Target: {TARGET}")
print(f"Number of model features: {len(FEATURE_COLS)}")
print(f"Appliance cols kept for reference: {len(APPLIANCE_COLS)}")

# Verify all feature columns exist
missing = [col for col in FEATURE_COLS if col not in df_features.columns]
print(f"Missing feature columns: {missing if missing else 'None — all present'}")

Target: use [kW]
Number of model features: 31
Appliance cols kept for reference: 15
Missing feature columns: None — all present


In [7]:
import os
import json

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save the full feature matrix (includes appliance cols for reference)
df_features.to_csv('../data/processed/feature_matrix.csv')

# Save the feature column list — this is critical for inference later
# The model must always receive features in exactly this order
with open('../models/feature_columns.json', 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)

print(f"Saved feature_matrix.csv — {df_features.shape}")
print(f"Saved feature_columns.json — {len(FEATURE_COLS)} features")
print(f"\nFile sizes:")
print(f"  feature_matrix.csv: {os.path.getsize('../data/processed/feature_matrix.csv')/1024:.1f} KB")
print(f"  feature_columns.json: {os.path.getsize('../models/feature_columns.json')/1024:.1f} KB")

Saved feature_matrix.csv — (8231, 47)
Saved feature_columns.json — 31 features

File sizes:
  feature_matrix.csv: 5589.8 KB
  feature_columns.json: 0.5 KB


## Feature Engineering

Builds the full feature matrix from cleaned hourly data.

**Input:** data/processed/hourly_data.csv (8,399 rows)  
**Output:** data/processed/feature_matrix.csv (8,231 rows, 47 columns)

**Features engineered (31 model inputs):**
- Time decomposition: hour, day_of_week, month, day_of_year, is_weekend, is_nighttime
- Cyclical encoding: hour_sin, hour_cos, day_sin, day_cos
- Lag features: 1h, 2h, 3h, 24h, 48h, 168h
- Rolling statistics: mean_3h, mean_24h, std_24h, max_24h
- Weather: temperature, humidity, windSpeed, cloudCover, precipIntensity,
  dewPoint, pressure, visibility, apparentTemperature, windBearing, precipProbability

**168 warmup rows dropped** — insufficient lag history at dataset start  
**Target variable:** use [kW] — total hourly household consumption